# Overnight → Intraday Reversal Screen (NQ, training slice)

Pre-registration: `research/hypotheses/overnight_effect.md` (locked in commit `b678aa1`).

**Rules of this notebook:**
- Training slice only (2018-01-02 → 2022-12-30). Test slice sealed.
- No parameter changes. Bucket count (5), horizons ({30m, 1h, 2h, close}), burn-in (252), bootstrap iterations (10k), significance level (95%), and pass criteria are all fixed by the pre-registration.
- Pass/fail is evaluated mechanically at the end. No negotiation.

In [ ]:
import sys
from pathlib import Path
from datetime import time as dt_time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make engine/ importable from notebook location
repo_root = Path.cwd().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.data_loader import load_csv, split_train_test

# Locked constants from pre-registration
N_BUCKETS = 5
BURN_IN_DAYS = 252
N_BOOTSTRAP = 10_000
RNG_SEED = 20260423  # session date; reproducible bootstrap
HALF_CUTOFF = pd.Timestamp("2020-07-01")  # stability-check split point

pd.set_option("display.float_format", lambda x: f"{x:+.5%}" if abs(x) < 1 else f"{x:.4f}")

## 1. Load and split
Use `engine.data_loader.load_csv` + `split_train_test`. Drop the test DataFrame from memory immediately so there's no accidental exposure.

In [ ]:
DATA_PATH = repo_root / "data" / "nq_15m_data.csv"
df_full = load_csv(str(DATA_PATH))
train, _test = split_train_test(df_full)
del df_full, _test  # test slice must not leak into this notebook

print(f"Training bars: {len(train):,}")
print(f"Range: {train.index.min()}  →  {train.index.max()}")
print(f"Timezone: {train.index.tz}")

## 2. Extract per-day RTH open and RTH close prices
- **RTH open price** = open of the 09:30 ET bar (bar is labeled at its start, so this is the true 09:30 ET print).
- **RTH close price** = close of the 15:45 ET bar (this is the 16:00 ET print, i.e. the cash-equities close).

Days where either bar is missing (early-close sessions, data gaps) are flagged and propagate as NaN. No ad-hoc fallback.

In [ ]:
t_open = dt_time(9, 30)
t_close = dt_time(15, 45)

bar_time = train.index.time
bar_date = train.index.date

open_mask = bar_time == t_open
close_mask = bar_time == t_close

open_df = pd.DataFrame({"date": bar_date[open_mask], "rth_open": train.loc[open_mask, "open"].values})
close_df = pd.DataFrame({"date": bar_date[close_mask], "rth_close": train.loc[close_mask, "close"].values})

daily = open_df.merge(close_df, on="date", how="outer").sort_values("date").reset_index(drop=True)
daily["date"] = pd.to_datetime(daily["date"])

n_total = len(daily)
n_missing_open = int(daily["rth_open"].isna().sum())
n_missing_close = int(daily["rth_close"].isna().sum())
print(f"Calendar rows (any of open/close present): {n_total}")
print(f"Missing 09:30 bar: {n_missing_open}")
print(f"Missing 15:45 bar: {n_missing_close}")
daily.head()

## 3. Compute `overnight_return`
$$\text{overnight\_return}_t = \frac{\text{rth\_open}_t - \text{rth\_close}_{t-1}}{\text{rth\_close}_{t-1}}$$

Shift happens on the date-sorted `daily` frame. Rows missing either side are NaN and dropped downstream.

In [ ]:
daily["prev_rth_close"] = daily["rth_close"].shift(1)
daily["overnight_return"] = (daily["rth_open"] - daily["prev_rth_close"]) / daily["prev_rth_close"]

n_valid = int(daily["overnight_return"].notna().sum())
print(f"Days with valid overnight_return: {n_valid}")
print()
print(daily["overnight_return"].describe())

## 4. Forward-return prices and returns at four locked horizons
Horizons from RTH open (09:30 ET price):
- 30 min → open of 10:00 ET bar
- 1 hour → open of 10:30 ET bar
- 2 hours → open of 11:30 ET bar
- close → rth_close (15:45 bar close, i.e. 16:00 ET print)

In [ ]:
def price_at(train_df, target_time, col):
    m = train_df.index.time == target_time
    return pd.DataFrame({
        "date": pd.to_datetime(train_df.index.date[m]),
        "price": train_df.loc[m, col].values,
    })

p30  = price_at(train, dt_time(10, 0),  "open").rename(columns={"price": "p_30m"})
p1h  = price_at(train, dt_time(10, 30), "open").rename(columns={"price": "p_1h"})
p2h  = price_at(train, dt_time(11, 30), "open").rename(columns={"price": "p_2h"})

for extra in (p30, p1h, p2h):
    daily = daily.merge(extra, on="date", how="left")

daily["fwd_30m"]   = (daily["p_30m"]   - daily["rth_open"]) / daily["rth_open"]
daily["fwd_1h"]    = (daily["p_1h"]    - daily["rth_open"]) / daily["rth_open"]
daily["fwd_2h"]    = (daily["p_2h"]    - daily["rth_open"]) / daily["rth_open"]
daily["fwd_close"] = (daily["rth_close"] - daily["rth_open"]) / daily["rth_open"]

HORIZONS = ["fwd_30m", "fwd_1h", "fwd_2h", "fwd_close"]

print("Missing count per horizon:")
print(daily[HORIZONS].isna().sum())
print()
print("Unconditional stats per horizon:")
daily[HORIZONS].describe().loc[["count", "mean", "std", "min", "max"]]

## 5. Expanding-percentile bucketing with 252-day burn-in
For each day $i$ (with $i \ge \text{BURN\_IN\_DAYS}$), compute quintile edges from the overnight-return values on days $[0, i)$ — strictly prior. Assign day $i$ to the bucket implied by those edges. This prevents day $i$'s bucket from depending on any future information.

In [ ]:
# Work only on rows with a valid overnight_return for the expanding series
valid = daily.dropna(subset=["overnight_return"]).reset_index(drop=True).copy()
ovn = valid["overnight_return"].values

buckets = np.full(len(valid), np.nan)
for i in range(BURN_IN_DAYS, len(valid)):
    history = ovn[:i]
    edges = np.quantile(history, np.linspace(0, 1, N_BUCKETS + 1))
    # Inner edges define bucket boundaries; searchsorted gives 0..N_BUCKETS-1
    buckets[i] = np.searchsorted(edges[1:-1], ovn[i], side="right")

valid["bucket"] = buckets
scored = valid.dropna(subset=["bucket"]).copy()
scored["bucket"] = scored["bucket"].astype(int)

print(f"Burn-in days excluded: {BURN_IN_DAYS}")
print(f"Days with assigned bucket: {len(scored)}")
print()
print("Bucket counts:")
print(scored["bucket"].value_counts().sort_index().rename(index=lambda b: f"Q{b+1}"))

## 6. De-mean forward returns against unconditional mean
Per-horizon unconditional mean computed on post-burn-in training days (consistent evaluation window). De-meaning strips unconditional drift so bucket means reflect *conditional* effect only.

In [ ]:
DM = {}
for h in HORIZONS:
    mu = scored[h].mean()
    DM[h] = f"{h}_dm"
    scored[DM[h]] = scored[h] - mu
    print(f"{h}: unconditional mean = {mu:+.5%}")

DM_HORIZONS = [DM[h] for h in HORIZONS]

## 7. Main bucket table (full training slice, post burn-in)
5 buckets × 4 horizons. Means are de-meaned forward returns; `n` is per-bucket day count.

In [ ]:
def bucket_means(df, horizons_dm):
    rows = []
    for b in range(N_BUCKETS):
        sub = df[df["bucket"] == b]
        row = {"bucket": f"Q{b+1}", "n": len(sub)}
        for h in horizons_dm:
            row[h] = sub[h].mean()
        rows.append(row)
    return pd.DataFrame(rows)

main_table = bucket_means(scored, DM_HORIZONS)
main_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(N_BUCKETS)
for h in DM_HORIZONS:
    ax.plot(x, main_table[h].values, marker="o", label=h.replace("_dm", ""))
ax.axhline(0, color="k", linewidth=0.5, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}" for i in range(N_BUCKETS)])
ax.set_ylabel("De-meaned forward return")
ax.set_xlabel("Overnight-return bucket (Q1 = biggest gap down, Q5 = biggest gap up)")
ax.set_title("NQ training slice: forward return by overnight-return bucket")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Bootstrap 95% confidence intervals (day-level resampling)
For each bucket × horizon cell: 10,000 iterations, each resampling the day-level values within that bucket with replacement. The CI is the 2.5/97.5 percentiles of the resampled mean distribution. Seed is fixed for reproducibility.

In [ ]:
def bootstrap_mean_ci(values, n_iter, rng):
    v = values[~np.isnan(values)]
    if len(v) == 0:
        return (np.nan, np.nan)
    idx = rng.integers(0, len(v), size=(n_iter, len(v)))
    boot_means = v[idx].mean(axis=1)
    return tuple(np.quantile(boot_means, [0.025, 0.975]))

def bucket_ci_table(df, horizons_dm):
    rng = np.random.default_rng(RNG_SEED)
    rows = []
    for b in range(N_BUCKETS):
        sub = df[df["bucket"] == b]
        row = {"bucket": f"Q{b+1}", "n": len(sub)}
        for h in horizons_dm:
            vals = sub[h].values
            lo, hi = bootstrap_mean_ci(vals, N_BOOTSTRAP, rng)
            mean = np.nanmean(vals) if len(vals) else np.nan
            excludes_zero = (lo > 0) or (hi < 0)
            row[f"{h}_mean"] = mean
            row[f"{h}_lo"] = lo
            row[f"{h}_hi"] = hi
            row[f"{h}_sig"] = "*" if excludes_zero else ""
        rows.append(row)
    return pd.DataFrame(rows)

ci_table = bucket_ci_table(scored, DM_HORIZONS)
ci_table

## 9. Stability check — Half A vs Half B
Date-based split at 2020-07-01 (as locked in pre-registration):
- Half A: 2018-01 → 2020-06
- Half B: 2020-07 → 2022-12

**Important**: bucket assignments for each day were computed once on the full training sequence (expanding percentiles). For the stability check we do not re-bucket within each half; we partition the already-bucketed days. This preserves the pre-registered definition.

In [ ]:
half_a = scored[scored["date"] < HALF_CUTOFF].copy()
half_b = scored[scored["date"] >= HALF_CUTOFF].copy()

print(f"Half A: {len(half_a)} days, {half_a['date'].min().date()} → {half_a['date'].max().date()}")
print(f"Half B: {len(half_b)} days, {half_b['date'].min().date()} → {half_b['date'].max().date()}")

a_table = bucket_means(half_a, DM_HORIZONS)
b_table = bucket_means(half_b, DM_HORIZONS)

print("\nHalf A bucket means:")
display(a_table)
print("\nHalf B bucket means:")
display(b_table)

## 10. Mechanical pass/fail evaluation
From the pre-registration, all four criteria must hold on the **1-hour horizon** (primary):

1. Strictly decreasing Q1 → Q5 means (matches reversal prediction)
2. Q1 and Q5 bootstrap 95% CIs both exclude zero
3. Q1 mean > 0 and Q5 mean < 0 (signs match prediction)
4. Strictly decreasing pattern holds in both Half A and Half B independently

In [ ]:
PRIMARY = "fwd_1h_dm"

def strictly_decreasing(arr):
    return bool(np.all(np.diff(arr) < 0))

# Criterion 1: monotonic decreasing on full slice
full_means = main_table[PRIMARY].values
crit1 = strictly_decreasing(full_means)

# Criterion 2: Q1 and Q5 CIs both exclude zero
q1 = ci_table.iloc[0]
q5 = ci_table.iloc[-1]
q1_excludes = (q1[f"{PRIMARY}_lo"] > 0) or (q1[f"{PRIMARY}_hi"] < 0)
q5_excludes = (q5[f"{PRIMARY}_lo"] > 0) or (q5[f"{PRIMARY}_hi"] < 0)
crit2 = bool(q1_excludes and q5_excludes)

# Criterion 3: signs match
q1_mean = q1[f"{PRIMARY}_mean"]
q5_mean = q5[f"{PRIMARY}_mean"]
crit3 = bool((q1_mean > 0) and (q5_mean < 0))

# Criterion 4: monotonic in both halves
crit4 = strictly_decreasing(a_table[PRIMARY].values) and strictly_decreasing(b_table[PRIMARY].values)

print("Pre-registered pass criteria (evaluated on fwd_1h_dm, the primary horizon):\n")
print(f"  1. Strictly decreasing Q1→Q5 on full slice:        {crit1}")
print(f"       means: {', '.join(f'{v:+.5%}' for v in full_means)}")
print(f"  2. Q1 and Q5 bootstrap 95% CIs exclude zero:       {crit2}")
print(f"       Q1 CI: [{q1[f'{PRIMARY}_lo']:+.5%}, {q1[f'{PRIMARY}_hi']:+.5%}]  excludes 0: {q1_excludes}")
print(f"       Q5 CI: [{q5[f'{PRIMARY}_lo']:+.5%}, {q5[f'{PRIMARY}_hi']:+.5%}]  excludes 0: {q5_excludes}")
print(f"  3. Q1 positive and Q5 negative:                    {crit3}")
print(f"       Q1 mean: {q1_mean:+.5%},  Q5 mean: {q5_mean:+.5%}")
print(f"  4. Strictly decreasing in both halves:             {crit4}")
print(f"       Half A means: {', '.join(f'{v:+.5%}' for v in a_table[PRIMARY].values)}")
print(f"       Half B means: {', '.join(f'{v:+.5%}' for v in b_table[PRIMARY].values)}")

overall = crit1 and crit2 and crit3 and crit4
print("\n" + ("OVERALL: PASS" if overall else "OVERALL: FAIL"))
if not overall:
    binding = []
    if not crit1: binding.append("1 (monotonicity on full slice)")
    if not crit2: binding.append("2 (extreme-bucket significance)")
    if not crit3: binding.append("3 (predicted signs)")
    if not crit4: binding.append("4 (stability across halves)")
    print(f"   Binding criterion failure(s): {', '.join(binding)}")

## 11. Recording outcome
Copy the numeric tables and the pass/fail verdict into the **Test log** section of `research/hypotheses/overnight_effect.md`. Do not edit the pre-registered sections above the Test log.

**If PASS:** the feature/bucketing/horizon set is frozen as-is. Next session, this same code path runs exactly once on the 2023–2024 test slice. No tuning between screen and test.

**If FAIL:** archive as-is, commit the populated test log, move on to a different hypothesis. Do not re-parameterize and re-run on the training slice.